In [1]:
!pip install s3fs

In [8]:
import pandas as pd
import boto3
import sagemaker
import json
from sklearn import datasets
from sklearn.model_selection import train_test_split

from sagemaker.estimator import Estimator
from sagemaker.model import Model

In [3]:
sess = sagemaker.Session()
region = sess.boto_region_name

s3_client = boto3.client("s3", region_name=region)

sagemaker_role = sagemaker.get_execution_role()

bucket = sess.default_bucket()
bucket_prefix = "iris-demo"

bucket, bucket_prefix, sagemaker_role, region

('sagemaker-ap-southeast-2-879381285437',
 'iris-demo',
 'arn:aws:iam::879381285437:role/service-role/AmazonSageMaker-ExecutionRole-20241108T211736',
 'ap-southeast-2')

In [4]:
base = f"s3://{bucket}/{bucket_prefix}"

train_path = f"{base}/data/train.csv"
test_path = f"{base}/data/test.csv"

model_dir = f"{base}/model"

In [5]:
iris = datasets.load_iris()

iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

train_df, test_df = train_test_split(iris_df, test_size=0.2)

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

train_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
49,5.0,3.3,1.4,0.2,0
23,5.1,3.3,1.7,0.5,0
16,5.4,3.9,1.3,0.4,0
108,6.7,2.5,5.8,1.8,2
26,5.0,3.4,1.6,0.4,0
...,...,...,...,...,...
120,6.9,3.2,5.7,2.3,2
86,6.7,3.1,4.7,1.5,1
109,7.2,3.6,6.1,2.5,2
76,6.8,2.8,4.8,1.4,1


In [6]:
image_uri = sagemaker.image_uris.retrieve(framework="xgboost", region=region, version="1.5-1")

estimator = Estimator(
    image_uri=image_uri,
    role=sagemaker_role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=model_dir,
    base_job_name="xgboost-custom-training",
    entry_point="train.py",
    source_dir=".",
    dependencies=["requirements.txt"]
)

estimator.set_hyperparameters(
    max_depth=6,
    eta=0.1,
    objective="multi:softmax",
    eval_metric="mlogloss",
    num_class=3,
    num_round=100,
    train_data=train_path,
    model_dir="/opt/ml/model"
)

estimator.fit()

INFO:sagemaker:Creating training-job with name: xgboost-custom-training-2024-11-10-01-19-20-513


2024-11-10 01:22:02 Starting - Starting the training job...
2024-11-10 01:22:17 Starting - Preparing the instances for training...
2024-11-10 01:22:39 Downloading - Downloading input data...
2024-11-10 01:23:14 Downloading - Downloading the training image......
2024-11-10 01:24:15 Training - Training image download completed. Training in progress./miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2024-11-10 01:24:20.728 ip-10-0-140-108.ap-southeast-2.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2024-11-10 01:24:20.756 ip-10-0-140-108.ap-southeast-2.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2024-11-10:01:24:21:INFO] Imported framework sagemaker_xgboost_container.training
[2024-11-10:01:24:21:INFO] No GPUs detect

In [35]:
model_artifact = f"{model_dir}/xgboost-custom-training-2024-11-10-01-19-20-513/output/model.tar.gz"

model = Model(
    image_uri=image_uri,
    model_data=model_artifact,
    role=sagemaker_role,
    entry_point="inference.py",
    source_dir=".",
    sagemaker_session=sess,
    dependencies=["requirements.txt"]
)

model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-ap-southeast-2-879381285437/iris-demo/model/xgboost-custom-training-2024-11-10-01-19-20-513/output/model.tar.gz), script artifact (.), and dependencies (['requirements.txt']) into single tar.gz file located at s3://sagemaker-ap-southeast-2-879381285437/sagemaker-xgboost-2024-11-10-05-02-45-046/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: sagemaker-xgboost-2024-11-10-05-05-36-758
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2024-11-10-05-05-37-438
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2024-11-10-05-05-37-438


---------!

In [41]:
input_data = [
    {
        "sepal length (cm)": 5.1,
        "sepal width (cm)": 3.5,
        "petal length (cm)": 1.4,
        "petal width (cm)": 0.2
    },
    {
        "sepal length (cm)": 6.7,
        "sepal width (cm)": 2.5,
        "petal length (cm)": 5.8,
        "petal width (cm)": 1.8
    },
]
payload_df = pd.DataFrame(input_data)
payload = payload_df.to_json(orient="records")

payload

'[{"sepal length (cm)":5.1,"sepal width (cm)":3.5,"petal length (cm)":1.4,"petal width (cm)":0.2},{"sepal length (cm)":6.7,"sepal width (cm)":2.5,"petal length (cm)":5.8,"petal width (cm)":1.8}]'

In [42]:
sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=region)

endpoint_name = "sagemaker-xgboost-2024-11-10-05-05-37-438"

response = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=payload
)

result = json.loads(response["Body"].read().decode())

result

[0.0, 2.0]